# 09 - Calibration: from classifier output to trust score

**CPU is fine. Run all; idempotent** (recomputes from saved predictions).

A raw classifier probability is not a trust score. Calibration on the
validation part - disjoint from both training and test - is what licenses
reading the number as a probability, and therefore what licenses setting
operating thresholds by cost. Without it, "trust score" is a renamed softmax
output and a reviewer will say so.

Three questions, on `fusion_c_fused` for both splits and all seeds:

1. **Which calibrator?** none / Platt / isotonic - Brier and ECE on test.
2. **Does one calibrator serve both regimes?** Calibration fitted on the whole
   validation set vs fitted *per regime* (no-certificate / certificate-holder).
   If per-regime calibration lowers ECE inside each regime, the score is
   adaptive in a precise sense: the same raw output means different things in
   different regimes and must be mapped differently.
3. **Operating points and the deferral band.** The score gates the
   deterministic DANE/TLSA check. Three thresholds are derived on validation
   and reported on test: high-recall, low-FPR, and the band between them where
   the score abstains and defers to cryptography. Band width and what falls
   into it are the operational cost of the design.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
from src.evaluate import metrics, predictions
from src.models.calibrate import Calibrator, reliability_curve
from src.utils import manifest as mf

PRED_DIR = Path(P['artifacts']['predictions']); TAB = Path(P['results']['tables'])
PRIMARY = 'fusion_c_fused'
SPLITS, SEEDS = ['family_disjoint_v1','random_v1'], [42,43,44]

def load_pair(split_name, seed):
    te = predictions.load(f'{PRIMARY}_{split_name}_s{seed}', PRED_DIR)
    va = predictions.load(f'{PRIMARY}_{split_name}_s{seed}_VAL', PRED_DIR)
    # val predictions were saved without the regime flag; recover it from the feature matrix
    hc = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet", columns=['domain','has_certificate'])
    va = va.merge(hc, on='domain', how='left')
    te['has_certificate'] = te['has_certificate'].astype(bool); va['has_certificate'] = va['has_certificate'].astype(bool)
    return va, te

va, te = load_pair('family_disjoint_v1', 42)
print('val', va.shape, '| test', te.shape)
print('regime sizes (test): cert', int(te.has_certificate.sum()), '| nocert', int((~te.has_certificate).sum()))

## 1. Which calibrator

In [ ]:
rows = []
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        for method in ['none','platt','isotonic']:
            cal = Calibrator(method).fit(va['raw_score'], va['true_label'])
            p = cal.transform(te['raw_score'])
            m = metrics.evaluate(te['true_label'], p)
            rows.append({'split': split_name, 'seed': seed, 'method': method,
                         'brier': m['brier_score'], 'ece': m['ece'], 'mce': m['mce'],
                         'roc_auc': m['roc_auc']})
cal_tab = (pd.DataFrame(rows).groupby(['split','method'])[['brier','ece','mce','roc_auc']]
             .agg(['mean','std']).round(4))
display(cal_tab)
cal_tab.to_csv(TAB/'table_calibration_methods.csv')
BEST = 'isotonic'
print('ROC unchanged by monotone calibration (as it must be); Brier/ECE are the decision.')

## 2. Global vs per-regime calibration

The adaptive question. Fit isotonic once on all of validation, or fit two
isotonic maps - one on validation no-certificate rows, one on validation
certificate-holders - and apply each to its regime in test. ECE and Brier are
then reported *inside each regime*.

In [ ]:
rows = []
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        # global
        g = Calibrator(BEST).fit(va['raw_score'], va['true_label'])
        p_global = g.transform(te['raw_score'])
        # per-regime
        p_regime = np.empty(len(te))
        for flag in (False, True):
            vmask, tmask = (va.has_certificate == flag).values, (te.has_certificate == flag).values
            if va.loc[vmask,'true_label'].nunique() < 2:
                p_regime[tmask] = g.transform(te.loc[tmask,'raw_score']); continue
            c = Calibrator(BEST).fit(va.loc[vmask,'raw_score'], va.loc[vmask,'true_label'])
            p_regime[tmask] = c.transform(te.loc[tmask,'raw_score'])
        for scheme, p in [('global', p_global), ('per_regime', p_regime)]:
            for reg, mask in [('all', np.ones(len(te), bool)),
                              ('nocert', ~te.has_certificate.values),
                              ('cert',    te.has_certificate.values)]:
                y, pp = te.loc[mask,'true_label'].values, p[mask]
                if len(np.unique(y)) < 2: continue
                m = metrics.evaluate(y, pp)
                rows.append({'split': split_name, 'seed': seed, 'scheme': scheme, 'regime': reg,
                             'ece': m['ece'], 'brier': m['brier_score'], 'mce': m['mce'],
                             'n': int(mask.sum()), 'prevalence': float(y.mean())})
regime_tab = (pd.DataFrame(rows).groupby(['split','regime','scheme'])[['ece','brier','mce']]
                .agg(['mean','std']).round(4))
display(regime_tab)
regime_tab.to_csv(TAB/'table_calibration_per_regime.csv')

d = pd.DataFrame(rows)
for split_name in SPLITS:
    for reg in ('nocert','cert'):
        g_ = d[(d.split==split_name)&(d.regime==reg)&(d.scheme=='global')]['ece'].mean()
        r_ = d[(d.split==split_name)&(d.regime==reg)&(d.scheme=='per_regime')]['ece'].mean()
        print(f'{split_name:20s} {reg:7s} ECE global={g_:.4f}  per-regime={r_:.4f}  delta={g_-r_:+.4f}')

## 3. Reliability diagrams (data for figures)

In [ ]:
for split_name in SPLITS:
    va, te = load_pair(split_name, 42)
    g = Calibrator(BEST).fit(va['raw_score'], va['true_label'])
    for name, p in [('raw', te['raw_score'].values), ('isotonic', g.transform(te['raw_score']))]:
        for reg, mask in [('all', np.ones(len(te), bool)),
                          ('nocert', ~te.has_certificate.values), ('cert', te.has_certificate.values)]:
            curve = pd.DataFrame(reliability_curve(te.loc[mask,'true_label'].values, p[mask], n_bins=15))
            curve.to_csv(TAB/f'reliability_{split_name}_{name}_{reg}.csv', index=False)
print('reliability curves written for figure generation')

## 4. Operating points and the deferral band

Thresholds are chosen on **validation** (calibrated scores) and then reported
on test. Three points:

* `t_high_recall` - the lowest score at which validation TPR reaches 95%.
  Below it, a domain is treated as low-suspicion and passes to normal
  resolution.
* `t_low_fpr` - the score at which validation FPR falls to 0.1%. Above it, a
  domain is high-suspicion and DANE/TLSA validation is mandatory before trust.
* between them: the **deferral band** - the score abstains and the
  deterministic cryptographic check decides. Its width and contents are the
  operational cost of the design, reported rather than hidden.

In [ ]:
from sklearn.metrics import roc_curve

def thresholds_from_val(va_scores, va_y, target_tpr=0.95, target_fpr=0.001):
    fpr, tpr, thr = roc_curve(va_y, va_scores)
    i_hr = np.argmax(tpr >= target_tpr)
    ok = np.where(fpr <= target_fpr)[0]
    i_lf = ok[np.argmax(tpr[ok])] if len(ok) else 0
    return float(thr[i_hr]), float(thr[i_lf])

rows = []
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        g = Calibrator(BEST).fit(va['raw_score'], va['true_label'])
        pv, pt = g.transform(va['raw_score']), g.transform(te['raw_score'])
        t_hr, t_lf = thresholds_from_val(pv, va['true_label'].values)
        lo, hi = min(t_hr, t_lf), max(t_hr, t_lf)
        y = te['true_label'].values
        pass_, defer, block = pt < lo, (pt >= lo) & (pt <= hi), pt > hi
        rows.append({
            'split': split_name, 'seed': seed, 't_low': lo, 't_high': hi,
            'pass_frac': pass_.mean(), 'defer_frac': defer.mean(), 'block_frac': block.mean(),
            'pass_malicious_rate': y[pass_].mean() if pass_.any() else np.nan,   # missed threats
            'block_benign_rate':   1-y[block].mean() if block.any() else np.nan, # collateral
            'defer_malicious_rate': y[defer].mean() if defer.any() else np.nan,
            'recall_block_only': (block & (y==1)).sum()/max((y==1).sum(),1),
            'recall_block_plus_defer': ((block|defer) & (y==1)).sum()/max((y==1).sum(),1),
            'defer_cert_share': te.loc[defer,'has_certificate'].mean() if defer.any() else np.nan,
        })
op = pd.DataFrame(rows)
op_tab = op.groupby('split').agg(['mean','std']).round(4)
display(op_tab[['t_low','t_high','pass_frac','defer_frac','block_frac']])
display(op_tab[['pass_malicious_rate','block_benign_rate','recall_block_only','recall_block_plus_defer','defer_cert_share']])
op.to_csv(TAB/'table_operating_points.csv', index=False)

## 5. Save the calibrated score

The primary trust score = isotonic-calibrated `fusion_c_fused`, saved per run
with both raw and calibrated columns so every later notebook (SHAP, figures)
reads one artifact.

In [ ]:
for split_name in SPLITS:
    for seed in SEEDS:
        va, te = load_pair(split_name, seed)
        g = Calibrator(BEST).fit(va['raw_score'], va['true_label'])
        run_id = f'trustscore_{split_name}_s{seed}'
        predictions.save(run_id, PRED_DIR, te['domain'].values, te['true_label'].values,
                         te['raw_score'].values, calibrated_score=g.transform(te['raw_score']),
                         extra={'has_certificate': te['has_certificate'].values})
        m = metrics.evaluate(te['true_label'], g.transform(te['raw_score']))
        mf.record(P['manifest'], run_id, 'trustscore', {'source_run': f'{PRIMARY}_{split_name}_s{seed}',
                  'calibration': BEST, 'fitted_on': 'validation part'}, split_name, None, m, seed, repo_root=REPO)
        print(f'{run_id:34s} ECE={m["ece"]:.4f} Brier={m["brier_score"]:.4f}')

---

**Reading section 4.** `pass_malicious_rate` is the threat that slips through
below the low threshold; `block_benign_rate` is collateral above the high one;
`defer_frac` is the share of traffic handed to the cryptographic check. The
design claim is that deferral is *targeted*: a small band that concentrates the
hard cases (`defer_malicious_rate` well above the base rate) rather than a
blanket fallback.

**Next:** `10_xai_evaluation` - SHAP on the fused model, explanation quality
(fidelity, stability, sparsity), and per-regime attribution.